In [1]:
import pandas as pd
import numpy as np
import joblib
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

In [2]:
df = pd.read_parquet("movies_tv_enriched.parquet")
df["content_type"] = "movie"

In [3]:
text_columns = ["title", "overview", "genres", "keywords", "cast", "director"]
for col in text_columns:
    df[col] = df[col].fillna("").astype(str).str.strip()

In [4]:
valid = (
    (df["vote_count"].fillna(0) >= 100) &
    (
        ((df["content_type"] == "movie") & (df["status"].fillna("").str.lower() == "released"))
    )
)

df["keywords_clean"] = df["keywords"].str.replace("|", " ", regex=False).str.replace(r"\s+", "_", regex=True)
df["genres_clean"] = df["genres"].str.replace("|", " ", regex=False).str.replace(r"\s+", "_", regex=True)

df["tfidf_text"] = (
    df["title"] + " " + df["title"] + " " +
    df["keywords_clean"] + " " + df["keywords_clean"] + " " +
    df["cast"] + " " +
    df["director"]
)

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english", ngram_range=(1,1 ), min_df=1, max_df=0.4,
    max_features=50000, sublinear_tf=True, dtype=np.float32
)
tfidf_matrix = tfidf_vectorizer.fit_transform(df["tfidf_text"])

indices = df.groupby(df["title"].fillna("").str.lower().str.strip()).groups

In [5]:
overview_embeddings = np.load("overview_embeddings (1).npy").astype(np.float32)
overview_ids = np.load("overview_ids (2).npy")
overview_content_types = np.load("overview_content_types.npy", allow_pickle=True)

composite_keys = list(zip(overview_ids, overview_content_types))
id_to_emb_pos = {key: pos for pos, key in enumerate(composite_keys)}

df_keys = list(zip(df["id"], df["content_type"]))
emb_positions = pd.Series(df_keys).map(id_to_emb_pos)
assert emb_positions.isna().sum() == 0, "Some df rows have no matching embedding"

overview_embeddings_aligned = overview_embeddings[emb_positions.to_numpy()]
print(overview_embeddings_aligned.shape, len(df))

(1246745, 384) 1246745


In [6]:
def is_same_franchise(title_a, title_b, n_words=2):
    def clean(t):
        words = str(t).lower().split()
        return [w for w in words if w not in {"the", "a", "an"}]

    a = clean(title_a)
    b = clean(title_b)

    a_prefix = " ".join(a[:n_words])
    b_prefix = " ".join(b[:n_words])

    return a_prefix and (a_prefix in b_prefix or b_prefix in a_prefix)
def clean_tokens(value):
    if pd.isna(value) or not str(value).strip():
        return set()
    value = str(value).lower().replace("|", " ")
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    return set(value.split())

def clean_keyword_tokens(value):
    if pd.isna(value) or not str(value).strip():
        return set()
    parts = str(value).lower().split("|")
    return {re.sub(r"[^a-z0-9\s]", "", p.strip()) for p in parts if p.strip()}

def top_cast(value, k=5):
    if pd.isna(value) or not str(value).strip():
        return set()
    names = str(value).split("|")[:k]
    cleaned = set()
    for n in names:
        n = re.sub(r"[^a-z0-9\s]", "", n.lower().strip())
        if n:
            cleaned.add(n)
    return cleaned


def hybrid_recommend(title, n=10, candidate_pool=300, content_type=None,
                      tfidf_weight=0.4, semantic_weight=0.6):

    title_clean = title.lower().strip()
    if title_clean not in indices:
        print(f"'{title}' not found.")
        return

    matching_indices = list(indices[title_clean])
    if content_type is not None:
        matching_indices = [i for i in matching_indices if df.iloc[i]["content_type"] == content_type]
        if not matching_indices:
            print(f"No {content_type} version of '{title}'.")
            return

    idx = max(matching_indices, key=lambda i: df.iloc[i]["vote_count"])

    tfidf_sims = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    semantic_sims = cosine_similarity(
        overview_embeddings_aligned[idx].reshape(1, -1),
        overview_embeddings_aligned
    ).flatten()

    sims = tfidf_weight * tfidf_sims + semantic_weight * semantic_sims

    candidate_indices = sims.argsort()[::-1]

    q_genre = clean_tokens(df.iloc[idx]["genres"])
    query_title = df.iloc[idx]["title"]
    
    candidates = [i for i in candidate_indices
                  if i not in matching_indices and valid.iloc[i]
                  and df.iloc[i]["vote_count"] >= 50
                  and (pd.isna(df.iloc[i].get("runtime")) or df.iloc[i]["runtime"] >= 40)
                  and (clean_tokens(df.iloc[i]["genres"]) & q_genre)][:candidate_pool]

    q_dir = clean_tokens(df.iloc[idx]["director"])
    q_cast = top_cast(df.iloc[idx]["cast"])
    q_kw = clean_keyword_tokens(df.iloc[idx]["keywords"])

    rows = []
    for i in candidates:
        m = df.iloc[i]
        sim = sims[i]

        director_bonus = 0.15 * sim if (q_dir & clean_tokens(m["director"])) else 0
        cast_bonus = 0.08 * sim * min(len(q_cast & top_cast(m["cast"])), 2)
        genre_bonus = 0.10 * sim * (len(q_genre & clean_tokens(m["genres"])) / max(len(q_genre), 1))
        kw_bonus = 0.05 * sim * min(len(q_kw & clean_keyword_tokens(m["keywords"])), 3)
        franchise_bonus = 0.35 if is_same_franchise(query_title, m["title"]) else 0
        
        quality_factor = min(math.log10(m["vote_count"] + 1) / math.log10(1000), 1)
        quality = min(m["vote_average"] / 10, 1) * 0.05 if m["vote_average"] else 0

        score = sim + (director_bonus + cast_bonus + genre_bonus + kw_bonus) * quality_factor + quality + franchise_bonus
        rows.append((i, score))

    rows.sort(key=lambda x: x[1], reverse=True)
    selected = rows[:n]
    sel_idx = [x[0] for x in selected]

    out = df.iloc[sel_idx][["title", "content_type", "vote_average", "vote_count", "release_date"]].copy()
    out["similarity"] = [sims[i] for i in sel_idx]
    out["hybrid_score"] = [x[1] for x in selected]
    return out.reset_index(drop=True)

In [7]:
df[["id", "overview"]].to_parquet("overview_only.parquet", compression="zstd")

In [8]:
emb16 = overview_embeddings_aligned.astype(np.float16)
np.save("overview_embeddings_aligned.npy", emb16)

In [9]:
needed_cols = ["id","title","content_type","vote_average","vote_count","release_date",
               "status","runtime","genres","keywords","cast","director"]
df[needed_cols].to_parquet("recommendation_df_slim.parquet", compression="zstd")

In [ ]:
joblib.dump(tfidf_matrix, "tfidf_matrix.joblib")
joblib.dump(indices, "title_indices.joblib")
joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.joblib")